In [ ]:
from threading import Thread
from multiprocessing import Process, Manager
import os
import json
import sys
import argparse
import datetime
import pandas as pd
from info_core import InitCore
# from exchange_plugin.common_info import InitCommonInfo
# from monitor_engine.kimp_core_monitor import InitKimpCoreMonitor
# from etc.db_handler.create_schema_tables import InitDBClient
# from trigger_engine.snatcher import InitSnatcher
from etc.register_monitor_msg import RegisterMonitorMsg
from etc.redis_connector.redis_connector import InitRedis
import _pickle as pickle
import time
import traceback

# for jupyter notebook
import nest_asyncio
nest_asyncio.apply()

In [ ]:
redis_cli = InitRedis(host='localhost', port=6379, db=0, passwd=None)

In [ ]:
temp = pickle.loads(redis_cli.get_data("INFO_CORE|UPBIT_SPOT/KRW:BINANCE_USD_M/USDT_1T_now"))
temp

In [ ]:
redis_cli.redis_conn.keys()

In [ ]:
logging_dir = "/home/info_core/"
with open("/home/info_core/info_core_config.json") as f:
    config = json.load(f)
proc_n = 2
# node = config['node']
node = 'kline_generator1_dev'
monitor_bot_name = config['monitor_setting']['monitor_bot']
monitor_bot_token = config['telegram_bot_setting'][monitor_bot_name]
monitor_bot_api_url = config['monitor_setting']['monitor_bot_api_url']
admin_id = config['telegram_admin_id']['charlie1155']
register_monitor_msg = RegisterMonitorMsg(monitor_bot_token, monitor_bot_api_url, admin_id, logging_dir)
master_flag = config['node_settings'][node]['MASTER']
# Read api keys
exchange_api_key_dict = config['exchange_api_key']
# Exchange market settings
enabled_market_klines = config['node_settings'][node]['enabled_market_klines']
# total enabled market settings
total_enabled_market_klines = []
for each_market_code_list in config['node_settings'].values():
    total_enabled_market_klines += each_market_code_list['enabled_market_klines']
total_enabled_market_klines = list(set(total_enabled_market_klines))

# db dict
db_dict = config['database_setting'][config['node_settings'][node]['db_settings']]

my_okx_demo_api_key = "bbb8a09a-6ea2-4686-add5-1095c918b7f4"
my_okx_demo_secret_key = "FBEAD86057449BAEC3FFA8A80CE76E4F"
my_okx_demo_passphrase = "121431Rn!!"

In [ ]:
core = InitCore(logging_dir, master_flag, proc_n, node, admin_id, register_monitor_msg, exchange_api_key_dict, enabled_market_klines, total_enabled_market_klines, db_dict)


In [ ]:
start = time.time()
target_market_code = 'UPBIT_SPOT/KRW'
origin_market_code = 'BINANCE_USD_M/USDT'
core.get_premium_df(target_market_code, origin_market_code)
time.time()-start

In [ ]:
def generate_ohlc_df(self, appended_df, freq='1T'):
    df = appended_df.set_index(['base_asset', 'datetime_now'])
    ohlc_df = pd.DataFrame()
    if 'tp_premium' in df.columns:
        df['tp_premium'] = pd.to_numeric(df['tp_premium'], errors='coerce')
        tp_ohlc_df = df.groupby(['base_asset', pd.Grouper(level='datetime_now', freq=freq)])['tp_premium'].ohlc()
        # add prefix of tp_ to the column names
        tp_ohlc_df.columns = ['tp_' + col for col in tp_ohlc_df.columns]
        ohlc_df = pd.concat([ohlc_df, tp_ohlc_df], axis=1)
    if 'LS_premium' in df.columns:
        df['LS_premium'] = pd.to_numeric(df['LS_premium'], errors='coerce')
        LS_ohlc_df = df.groupby(['base_asset', pd.Grouper(level='datetime_now', freq=freq)])['LS_premium'].ohlc()
        # add prefix of LS_ to the column names
        LS_ohlc_df.columns = ['LS_' + col for col in LS_ohlc_df.columns]
        ohlc_df = pd.concat([ohlc_df, LS_ohlc_df], axis=1)
    if 'SL_premium' in df.columns:
        df['SL_premium'] = pd.to_numeric(df['SL_premium'], errors='coerce')
        SL_ohlc_df = df.groupby(['base_asset', pd.Grouper(level='datetime_now', freq=freq)])['SL_premium'].ohlc()
        # add prefix of SL_ to the column names
        SL_ohlc_df.columns = ['SL_' + col for col in SL_ohlc_df.columns]
        ohlc_df = pd.concat([ohlc_df, SL_ohlc_df], axis=1)
    if 'tp_premium' not in df.columns and 'LS_premium' not in df.columns and 'SL_premium' not in df.columns:
        raise ValueError('There is no proper column in the dataframe')
    # TEST
    ohlc_df['dollar'] = df['dollar'].iloc[-1]
    ohlc_df.reset_index(inplace=True)
    # TEST
    try:
        ohlc_df['record_count'] = appended_df.groupby('base_asset')['base_asset'].count().values
    except Exception as e:
        content = f"Error in generate_ohlc_df: {traceback.format_exc()}\n ohlc_df:{ohlc_df}, appended_df.groupby('base_asset'):{appended_df.groupby('base_asset')}\n appended_df:{appended_df}"
        self.kline_logger.error(content)
    return ohlc_df

In [ ]:
def generate_1T_ohlc_df(self, old_premium_df, new_premium_df):
    ohlc_columns = ['open','high','low','close']
    price_name_list = ['tp','LS','SL']
    if len(old_premium_df) == 0:
        for each_ohlc_column in ohlc_columns:
            for each_price_name in price_name_list:
                if f"{each_price_name}_premium" in new_premium_df.columns:
                    new_premium_df[f"{each_price_name}_{each_ohlc_column}"] = new_premium_df[f"{each_price_name}_premium"]
    else:
        merged_df = pd.merge(old_premium_df, new_premium_df, on=['base_asset'], suffixed=('_old', '_new'))
        comparison_df = merged_df[['base_asset','quote_asset','tp','ap','bp','scr','atp24h','converted_tp','converted_ap','converted_bp','dollar']].copy()
        for column in ["tp_premium", 'LS_premium', 'SL_premium']:
            if column not in new_premium_df.columns:
                continue
            # compare
            
            
    # TEST
    ohlc_df['dollar'] = df['dollar'].iloc[-1]
    ohlc_df.reset_index(inplace=True)
    # TEST
    try:
        ohlc_df['record_count'] = appended_df.groupby('base_asset')['base_asset'].count().values
    except Exception as e:
        content = f"Error in generate_ohlc_df: {traceback.format_exc()}\n ohlc_df:{ohlc_df}, appended_df.groupby('base_asset'):{appended_df.groupby('base_asset')}\n appended_df:{appended_df}"
        self.kline_logger.error(content)
    return ohlc_df